In [1]:
%load_ext nb_black
import os
import glob
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pickle

from scipy import stats
from os.path import join
from pathlib import Path
import pandas as pd

<IPython.core.display.Javascript object>

In [2]:
cmip6_data_dir = Path.cwd().parent
res_dir = Path.cwd().parent / "processed"

<IPython.core.display.Javascript object>

In [3]:
# get all the variable file paths, get rid of the EC-Earth3-Veg becasue it is handled seperately
file_paths = [
    f
    for f in cmip6_data_dir.rglob("*.nc")
    if "sftlf" in f.name or "areacella" in f.name
]

print(len(file_paths))

217


<IPython.core.display.Javascript object>

In [4]:
sorted([i.name for i in file_paths])

['areacella_fx_ACCESS-CM2_1pctCO2_r1i1p1f1_gn.nc',
 'areacella_fx_ACCESS-CM2_historical_r1i1p1f1_gn.nc',
 'areacella_fx_ACCESS-CM2_piControl_r1i1p1f1_gn.nc',
 'areacella_fx_ACCESS-ESM1-5_1pctCO2_r1i1p1f1_gn.nc',
 'areacella_fx_ACCESS-ESM1-5_historical_r1i1p1f1_gn.nc',
 'areacella_fx_ACCESS-ESM1-5_piControl_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-CM-1-1-MR_1pctCO2_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-CM-1-1-MR_historical_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-CM-1-1-MR_piControl_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-ESM-1-1-LR_1pctCO2_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-ESM-1-1-LR_historical_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-ESM-1-1-LR_piControl_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-ESM-1-REcoM_historical_r1i1p1f1_gn.nc',
 'areacella_fx_AWI-ESM-1-REcoM_piControl_r1i1p1f1_gn.nc',
 'areacella_fx_BCC-CSM2-MR_hist-resIPO_r1i1p1f1_gn.nc',
 'areacella_fx_BCC-ESM1_1pctCO2_r1i1p1f1_gn.nc',
 'areacella_fx_CESM2-FV2_historical_r1i1p1f1_gn.nc',
 'areacella_fx_CESM2-FV2_piControl_r1i1p1f1_gn.nc',
 'areacella

<IPython.core.display.Javascript object>

In [5]:
data = []

for f in file_paths:
    parts = f.stem.split("_")
    variable = parts[0]
    mdl_exp_label = "_".join(parts[2:5])
    data.append([mdl_exp_label, variable, f])

df = pd.DataFrame(data, columns=["mdl_exp_label", "variable", "file_path"])
df = df.pivot(index="mdl_exp_label", columns="variable", values="file_path")
df.columns.name = None
df = df.dropna(subset=["areacella", "sftlf"])

<IPython.core.display.Javascript object>

In [6]:
preference = [
    "piControl",
    "1pctCO2",
    "historical",
    "hist-resIPO",
]

df = df.reset_index()
df["model"] = df["mdl_exp_label"].str.split("_").str[0]
df["experiment"] = df["mdl_exp_label"].str.split("_").str[1]

df["experiment_rank"] = df["experiment"].map(
    {exp: i for i, exp in enumerate(preference)}
)

df = (
    df.sort_values(["model", "experiment_rank"])
    .drop_duplicates("model")
    .drop(columns=["model", "experiment", "experiment_rank"])
    .set_index("mdl_exp_label")
)

<IPython.core.display.Javascript object>

In [7]:
df["model"] = [i.split("_")[0] for i in df.index]
df["mdl_exp_label"] = df.index
df = df.set_index("model")

<IPython.core.display.Javascript object>

In [8]:
import warnings
from xarray.coding.times import SerializationWarning

warnings.filterwarnings("ignore", category=SerializationWarning)

<IPython.core.display.Javascript object>

In [9]:
areacella_dict = {}
sftlf_dict = {}
land_area_dict = {}

for mdl, row in df.iterrows():
    areacella = xr.open_dataset(row["areacella"])["areacella"]
    sftlf = xr.open_dataset(row["sftlf"])["sftlf"]
    if "type" in sftlf.coords:
        sftlf = sftlf.reset_coords("type", drop=True)

    lat_identical = areacella.lat.equals(sftlf.lat)
    lon_identical = areacella.lon.equals(sftlf.lon)

    if not lat_identical or not lon_identical:
        print(
            mdl,
            "Lat/Lon not identical. Reindex sftlf to areacella coordinates using nearest neighbour",
        )
        sftlf = sftlf.reindex(
            lat=areacella.lat,
            lon=areacella.lon,
            method="nearest",
        )
        assert sftlf.lat.equals(areacella.lat) and sftlf.lon.equals(
            areacella.lon
        ), "Reindex failed, unmatch lat/lon not from numeric issue."

    areacella_dict[mdl] = areacella
    sftlf_dict[mdl] = sftlf
    land_area_dict[mdl] = areacella * sftlf / 100

AWI-CM-1-1-MR Lat/Lon not identical. Reindex sftlf to areacella coordinates using nearest neighbour
EC-Earth3-Veg Lat/Lon not identical. Reindex sftlf to areacella coordinates using nearest neighbour


<IPython.core.display.Javascript object>

In [10]:
results = []

for mdl in df.index:
    areacella = areacella_dict[mdl]
    sftlf = sftlf_dict[mdl]
    land_area = land_area_dict[mdl]

    lat = areacella["lat"]
    lon = areacella["lon"]

    results.append(
        {
            "model": mdl,
            "nlat": areacella.sizes["lat"],
            "nlon": areacella.sizes["lon"],
            "lat_range": f"{lat[0].item():.2f}–{lat[-1].item():.2f}",
            "lon_range": f"{lon[0].item():.2f}–{lon[-1].item():.2f}",
            "grid_area (10^14 m2)": areacella.sum().item() / 1e14,
            "land_fraction (%)": sftlf.mean().item(),
            "land_area (10^14 m2)": land_area.sum().item() / 1e14,
        }
    )

df_grid = pd.DataFrame(results).set_index("model")
df_grid

,nlat,nlon,lat_range,lon_range,grid_area (10^14 m2),land_fraction (%),land_area (10^14 m2)
model,,,,,,,
ACCESS-CM2,144,192,-89.38–89.38,0.94–359.06,5.100997,34.253513,1.487314
ACCESS-ESM1-5,145,192,-90.00–90.00,0.00–358.12,5.099146,34.352951,1.486614
AWI-CM-1-1-MR,192,384,-89.28–89.28,0.00–359.06,5.100645,33.805340,1.471893
AWI-ESM-1-1-LR,96,192,-88.57–88.57,0.00–358.12,5.100645,33.235676,1.446864
AWI-ESM-1-REcoM,96,192,-88.57–88.57,0.00–358.12,5.100645,33.235676,1.446864
BCC-CSM2-MR,160,320,-89.14–89.14,0.00–358.88,5.100650,34.320370,1.497911
BCC-ESM1,64,128,-87.86–87.86,0.00–357.19,5.100650,34.253494,1.497823
CESM2,192,288,-90.00–90.00,0.00–358.75,5.100997,34.495831,1.494484
CESM2-FV2,96,144,-90.00–90.00,0.00–357.50,5.100997,34.576870,1.494516


<IPython.core.display.Javascript object>

In [11]:
# save to pkl for reuse later
with open(res_dir / "land_area_related" / "land_area_dict.pkl", "wb") as f:
    pickle.dump(land_area_dict, f)

with open(res_dir / "land_area_related" / "areacella_dict.pkl", "wb") as f:
    pickle.dump(areacella_dict, f)

with open(res_dir / "land_area_related" / "sftlf_dict.pkl", "wb") as f:
    pickle.dump(sftlf_dict, f)

<IPython.core.display.Javascript object>